In [1]:
import tensorflow as tf
import os
import glob
import numpy as np
from model import build_1d_cnn_model
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES, tf_load_sample_from_files
cnn_model=build_1d_cnn_model(None,INPUT_SHAPE,OUTPUT_DIM_NOTES,False)
cnn_model.summary()
tf.keras.utils.plot_model(cnn_model,to_file='cnn_model.png',show_shapes=True)
cnn_model.load_weights('guitarmidi.weights.h5')

input_data_dir = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices/training'
input_filepaths = sorted(glob.glob(os.path.join(input_data_dir, '**', 'input', '*.npy'), recursive=True))
train_dataset = tf.data.Dataset.from_tensor_slices((input_filepaths))
train_dataset=train_dataset.shuffle(buffer_size=len(input_filepaths))
train_dataset=train_dataset.take(100)
train_dataset = train_dataset.map(tf_load_sample_from_files, num_parallel_calls=tf.data.AUTOTUNE)
def representative_data_gen():
  """A generator function that yields input data for quantization."""
  for input_value in train_dataset.batch(1).prefetch(tf.data.AUTOTUNE):
    # TFLite converter expects a list of input tensors
    yield [input_value[0]]

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)

converter.optimizations = [ tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.target_spec.supported_types = [tf.int8] 

# converter.target_spec.supported_types = [tf.float16]
# converter.target_spec._experimental_supported_accumulation_type = tf.dtypes.float16
# Perform the conversion
tflite_model = converter.convert()


# converter=tf.lite.TFLiteConverter.from_keras_model(cnn_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model=converter.convert()

with open('guitarmidi.tflite','wb') as f:
    f.write(tflite_model)

I0000 00:00:1767574789.636841  155292 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1767574789.665104  155292 cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1767574790.157859  155292 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Initial input shape: (None, 312, 1)


W0000 00:00:1767574790.688129  155292 gpu_device.cc:2456] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1767574790.693425  155292 gpu_device.cc:2456] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1767574790.764117  155292 gpu_device.cc:2040] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 11980 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5080, pci bus id: 0000:01:00.0, compute capability: 12.0a


After first Conv2D: (None, 312, 64)
Extracting string 1 from filters 0 to 52
String 1 section shape: (None, 52, 64)
String 1 after first Conv1D: (None, 52, 128)
String 1 after second Conv1D: (None, 13, 256)
Extracting string 2 from filters 52 to 104
String 2 section shape: (None, 52, 64)
String 2 after first Conv1D: (None, 52, 128)
String 2 after second Conv1D: (None, 13, 256)
Extracting string 3 from filters 104 to 156
String 3 section shape: (None, 52, 64)
String 3 after first Conv1D: (None, 52, 128)
String 3 after second Conv1D: (None, 13, 256)
Extracting string 4 from filters 156 to 208
String 4 section shape: (None, 52, 64)
String 4 after first Conv1D: (None, 52, 128)
String 4 after second Conv1D: (None, 13, 256)
Extracting string 5 from filters 208 to 260
String 5 section shape: (None, 52, 64)
String 5 after first Conv1D: (None, 52, 128)
String 5 after second Conv1D: (None, 13, 256)
Extracting string 6 from filters 260 to 312
String 6 section shape: (None, 52, 64)
String 6 after 

Model: "functional"

...

 Non-trainable params: 4,800 (18.75 KB)

INFO:tensorflow:Assets written to: /tmp/tmph9oet9o4/assets


INFO:tensorflow:Assets written to: /tmp/tmph9oet9o4/assets


Saved artifact at '/tmp/tmph9oet9o4'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 312, 256, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 129), dtype=tf.float32, name=None)
Captures:
  128548183812496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128548183814800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128548183814224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128548183814608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128548183812688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128548183814416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128548183814992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128548183815952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128548183815568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128548183815376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128548183813

/home/gerald/anaconda3/envs/gpu/lib/python3.13/site-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1767574793.745101  155292 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1767574793.745116  155292 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1767574793.745301  155292 reader.cc:83] Reading SavedModel from: /tmp/tmph9oet9o4
I0000 00:00:1767574793.746673  155292 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1767574793.746677  155292 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmph9oet9o4
I0000 00:00:1767574793.759032  155292 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1767574793.761501  155292 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1767574793.849616  155292 loader.cc:220] Running initialization op on S